# Auto Tagging Support Tickets Using LLM

## 1. Problem Statement
The goal of this project is to automatically classify support tickets into categories using a large language model and compare zero-shot and few-shot learning approaches.

## 2. Dataset Loading

## 3. Zero-shot Classification

## 4. Few-shot Classification

## 5. Evaluation and Comparison

## 6. Top 3 Tag Prediction

## 7. Deployment with Gradio

## 8. Final Summary

In [1]:
!pip install transformers datasets scikit-learn pandas numpy torch gradio

     ------------------------------------ 527.0/527.0 kB 516.8 kB/s eta 0:00:00
     ---------------------------------------- 43.0/43.0 MB 3.3 MB/s eta 0:00:00
     -------------------------------------- 27.5/27.5 MB 397.7 kB/s eta 0:00:00
     ------------------------------------ 120.0/120.0 kB 638.2 kB/s eta 0:00:00
  Using cached xxhash-3.6.0-cp311-cp311-win_amd64.whl (31 kB)
     ------------------------------------ 144.5/144.5 kB 330.6 kB/s eta 0:00:00
     ------------------------------------ 369.0/369.0 kB 410.1 kB/s eta 0:00:00
     ------------------------------------ 117.4/117.4 kB 380.0 kB/s eta 0:00:00
     -------------------------------------- 59.2/59.2 kB 627.1 kB/s eta 0:00:00
     ------------------------------------ 124.9/124.9 kB 386.2 kB/s eta 0:00:00
     -------------------------------------- 74.3/74.3 kB 820.9 kB/s eta 0:00:00
     -------------------------------------- 68.8/68.8 kB 340.6 kB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import pipeline

C:\Users\ayesh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
data = {
    "ticket_text": [
        "My internet connection is very slow and keeps disconnecting",
        "I was charged twice for my monthly subscription",
        "I cannot log into my account even after resetting my password",
        "The app crashes every time I try to upload a file",
        "Please cancel my subscription immediately",
        "I want a refund because I was billed incorrectly",
        "The website is showing a server error",
        "I forgot my password and cannot access my dashboard",
        "My payment did not go through but money was deducted",
        "The mobile app is not opening on my phone",
        "How do I change my plan from basic to premium?",
        "My order has not arrived yet and tracking is not updating"
    ],
    "true_tag": [
        "technical issue",
        "billing issue",
        "account access",
        "technical issue",
        "subscription",
        "billing issue",
        "technical issue",
        "account access",
        "billing issue",
        "technical issue",
        "subscription",
        "delivery issue"
    ]
}

df = pd.DataFrame(data)
df.head()

,ticket_text,true_tag
0,My internet connection is very slow and keeps ...,technical issue
1,I was charged twice for my monthly subscription,billing issue
2,I cannot log into my account even after resett...,account access
3,The app crashes every time I try to upload a file,technical issue
4,Please cancel my subscription immediately,subscription


In [19]:
candidate_labels = [
    "technical issue",
    "billing issue",
    "account access",
    "subscription",
    "delivery issue"
]

In [20]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

Device set to use cpu


In [21]:
def predict_top3_zero_shot(text, labels):
    result = classifier(text, candidate_labels=labels, multi_label=False)
    
    top_3_labels = result["labels"][:3]
    top_3_scores = result["scores"][:3]
    
    return list(zip(top_3_labels, top_3_scores))

In [22]:
sample_text = "I cannot access my profile because my password is not working"
predict_top3_zero_shot(sample_text, candidate_labels)

[('account access', 0.5690260529518127),
 ('technical issue', 0.38364219665527344),
 ('delivery issue', 0.034518685191869736)]

In [23]:
zero_shot_predictions = []
zero_shot_top3 = []

for text in df["ticket_text"]:
    preds = predict_top3_zero_shot(text, candidate_labels)
    zero_shot_top3.append(preds)
    zero_shot_predictions.append(preds[0][0])

df["zero_shot_pred"] = zero_shot_predictions
df["zero_shot_top3"] = zero_shot_top3

df.head()

,ticket_text,true_tag,zero_shot_pred,zero_shot_top3
0,My internet connection is very slow and keeps ...,technical issue,technical issue,"[(technical issue, 0.772585928440094), (accoun..."
1,I was charged twice for my monthly subscription,billing issue,subscription,"[(subscription, 0.5160489678382874), (billing ..."
2,I cannot log into my account even after resett...,account access,account access,"[(account access, 0.8940738439559937), (techni..."
3,The app crashes every time I try to upload a file,technical issue,technical issue,"[(technical issue, 0.6979901194572449), (deliv..."
4,Please cancel my subscription immediately,subscription,subscription,"[(subscription, 0.8295723795890808), (billing ..."


In [24]:
zero_shot_acc = accuracy_score(df["true_tag"], df["zero_shot_pred"])
zero_shot_f1 = f1_score(df["true_tag"], df["zero_shot_pred"], average="weighted")

print("Zero-shot Accuracy:", zero_shot_acc)
print("Zero-shot F1-score:", zero_shot_f1)

print("\nClassification Report:\n")
print(classification_report(df["true_tag"], df["zero_shot_pred"]))

Zero-shot Accuracy: 0.8333333333333334
Zero-shot F1-score: 0.8333333333333334

Classification Report:

                 precision    recall  f1-score   support

 account access       1.00      1.00      1.00         2
  billing issue       0.67      0.67      0.67         3
 delivery issue       1.00      1.00      1.00         1
   subscription       0.50      0.50      0.50         2
technical issue       1.00      1.00      1.00         4

       accuracy                           0.83        12
      macro avg       0.83      0.83      0.83        12
   weighted avg       0.83      0.83      0.83        12



In [25]:
label_descriptions = {
    "technical issue": "Examples: app crash, website error, internet problem, upload failure, server issue",
    "billing issue": "Examples: charged twice, refund request, incorrect billing, payment deducted",
    "account access": "Examples: login problem, password reset, account locked, cannot access dashboard",
    "subscription": "Examples: cancel subscription, upgrade plan, downgrade plan, change package",
    "delivery issue": "Examples: order not arrived, delayed shipment, tracking not updating, package missing"
}

In [26]:
def build_few_shot_text(ticket_text):
    context = "Support ticket categories:\n"
    for label, desc in label_descriptions.items():
        context += f"- {label}: {desc}\n"
    context += f"\nTicket: {ticket_text}"
    return context

In [27]:
def predict_top3_few_shot(text, labels):
    enriched_text = build_few_shot_text(text)
    result = classifier(enriched_text, candidate_labels=labels, multi_label=False)
    
    top_3_labels = result["labels"][:3]
    top_3_scores = result["scores"][:3]
    
    return list(zip(top_3_labels, top_3_scores))

In [32]:
predict_top3_few_shot("I was billed twice this month and need a refund", candidate_labels)

[('billing issue', 0.3633154332637787),
 ('technical issue', 0.3373092710971832),
 ('account access', 0.12624391913414001)]

In [36]:
few_shot_predictions = []
few_shot_top3 = []

for text in df["ticket_text"]:
    preds = predict_top3_few_shot(text, candidate_labels)
    few_shot_top3.append(preds)
    few_shot_predictions.append(preds[0][0])

df["few_shot_pred"] = few_shot_predictions
df["few_shot_top3"] = few_shot_top3

df.head()

,ticket_text,true_tag,zero_shot_pred,zero_shot_top3,few_shot_pred,few_shot_top3
0,My internet connection is very slow and keeps ...,technical issue,technical issue,"[(technical issue, 0.772585928440094), (accoun...",technical issue,"[(technical issue, 0.5421267151832581), (accou..."
1,I was charged twice for my monthly subscription,billing issue,subscription,"[(subscription, 0.5160489678382874), (billing ...",billing issue,"[(billing issue, 0.333524614572525), (technica..."
2,I cannot log into my account even after resett...,account access,account access,"[(account access, 0.8940738439559937), (techni...",account access,"[(account access, 0.4281228184700012), (techni..."
3,The app crashes every time I try to upload a file,technical issue,technical issue,"[(technical issue, 0.6979901194572449), (deliv...",technical issue,"[(technical issue, 0.7552521824836731), (accou..."
4,Please cancel my subscription immediately,subscription,subscription,"[(subscription, 0.8295723795890808), (billing ...",technical issue,"[(technical issue, 0.4054335057735443), (subsc..."


In [37]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

few_shot_acc = accuracy_score(df["true_tag"], df["few_shot_pred"])
few_shot_f1 = f1_score(df["true_tag"], df["few_shot_pred"], average="weighted")

print("Few-shot Accuracy:", few_shot_acc)
print("Few-shot F1-score:", few_shot_f1)

print("\nClassification Report:\n")
print(classification_report(df["true_tag"], df["few_shot_pred"]))

Few-shot Accuracy: 0.75
Few-shot F1-score: 0.6924242424242424

Classification Report:

                 precision    recall  f1-score   support

 account access       1.00      1.00      1.00         2
  billing issue       1.00      0.67      0.80         3
 delivery issue       1.00      1.00      1.00         1
   subscription       0.00      0.00      0.00         2
technical issue       0.57      1.00      0.73         4

       accuracy                           0.75        12
      macro avg       0.71      0.73      0.71        12
   weighted avg       0.69      0.75      0.69        12



C:\Users\ayesh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ayesh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ayesh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [39]:
comparison = pd.DataFrame({
    "Method": ["Zero-shot", "Few-shot"],
    "Accuracy": [zero_shot_acc, few_shot_acc],
    "Weighted F1-score": [zero_shot_f1, few_shot_f1]
})

comparison

,Method,Accuracy,Weighted F1-score
0,Zero-shot,0.833333,0.833333
1,Few-shot,0.750000,0.692424


In [40]:
for i in range(len(df)):
    print(f"Ticket: {df.loc[i, 'ticket_text']}")
    print(f"True Tag: {df.loc[i, 'true_tag']}")
    print(f"Zero-shot Top 3: {df.loc[i, 'zero_shot_top3']}")
    print(f"Few-shot Top 3: {df.loc[i, 'few_shot_top3']}")
    print("-" * 80)

Ticket: My internet connection is very slow and keeps disconnecting
True Tag: technical issue
Zero-shot Top 3: [('technical issue', 0.772585928440094), ('account access', 0.1404372602701187), ('delivery issue', 0.04622289538383484)]
Few-shot Top 3: [('technical issue', 0.5421267151832581), ('account access', 0.17130860686302185), ('delivery issue', 0.11553052812814713)]
--------------------------------------------------------------------------------
Ticket: I was charged twice for my monthly subscription
True Tag: billing issue
Zero-shot Top 3: [('subscription', 0.5160489678382874), ('billing issue', 0.41963258385658264), ('account access', 0.04586881771683693)]
Few-shot Top 3: [('billing issue', 0.333524614572525), ('technical issue', 0.29567912220954895), ('account access', 0.14149844646453857)]
--------------------------------------------------------------------------------
Ticket: I cannot log into my account even after resetting my password
True Tag: account access
Zero-shot Top 3

In [41]:
df.to_csv("ticket_tagging_results.csv", index=False)
print("Results saved as ticket_tagging_results.csv")

Results saved as ticket_tagging_results.csv


In [42]:
import gradio as gr

def live_ticket_tagger(text, mode):
    if mode == "Zero-shot":
        preds = predict_top3_zero_shot(text, candidate_labels)
    else:
        preds = predict_top3_few_shot(text, candidate_labels)
    
    output = ""
    for i, (label, score) in enumerate(preds, start=1):
        output += f"{i}. {label} ({score:.4f})\n"
    return output

demo = gr.Interface(
    fn=live_ticket_tagger,
    inputs=[
        gr.Textbox(lines=4, label="Enter Support Ticket"),
        gr.Radio(["Zero-shot", "Few-shot"], value="Zero-shot", label="Mode")
    ],
    outputs=gr.Textbox(label="Top 3 Predicted Tags"),
    title="LLM-Based Support Ticket Auto Tagging",
    description="Classify support tickets into categories using zero-shot and few-shot learning."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [44]:
print("""
Summary:
- Built an LLM-based support ticket tagging system.
- Applied zero-shot classification using facebook/bart-large-mnli.
- Improved label guidance using few-shot style prompting.
- Compared zero-shot and few-shot performance using Accuracy and F1-score.
- Returned the top 3 most probable tags for each ticket.
- Deployed the model using Gradio for live interaction.
""")


Summary:
- Built an LLM-based support ticket tagging system.
- Applied zero-shot classification using facebook/bart-large-mnli.
- Improved label guidance using few-shot style prompting.
- Compared zero-shot and few-shot performance using Accuracy and F1-score.
- Returned the top 3 most probable tags for each ticket.
- Deployed the model using Gradio for live interaction.

